# colab_13 — Geneformer CPT eval #3 / detector #2 (catastrophic forgetting)

Did the aggregated CPT run (colab_11 v2) **overwrite the base model's general cell-type knowledge** while adapting to AD glia? This is the other half of the CPT verdict: colab_12 asks whether CPT *helped* the biology (substate, APOE); this notebook asks whether it *hurt* what the model already knew.

**Eval #3 and detector #2 are the same measurement**, read two ways: k-NN cell-type accuracy on a non-AD reference, zero-shot vs post-CPT. The *drop* is scored against eval #3's reporting bands (≤5% acceptable / 5–15% concerning / >15% catastrophic) **and** detector #2's hard gate (>20% drop = overtrain).

**This gates colab_12.** Detector #2 is a go/no-go on the whole run: a substate or APOE "win" in colab_12 that coincides with detector #2 tripping is logged as **overtrain-confounded**, not a win. Section 6 applies that gate to colab_12's recorded verdicts.

**Same two extraction points as colab_12.** The reference is embedded at both `emb_layer=-1` (pipeline readout) and `emb_layer=0` (last layer). Forgetting can hide at one and show at the other — the head-ablation result showed v2's changes concentrate downstream of −1, so reading only −1 could miss damage that L0 reveals.

## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment, set run flags

Identical lean Geneformer-only stack as colab_09/11/12 (native numpy-2 base; scGPT not installed). A GPU is required — this notebook embeds the reference four times. `SMOKE` (committed `False`) subsamples the reference at 2b, *before* tokenization and the four embedding passes, so a plumbing run is genuinely cheap; it thins per (cell type x donor) so the donor-disjoint split at 5a is still exercised on real structure. Every derived path is `_SMOKE`-suffixed.

In [1]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

!pip install -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
# Pin the clone. pip pins cover PyPI packages, but Geneformer (library code AND the V2-104M
# weights) comes from this HF clone, which pip never sees -- an unpinned clone would let both
# drift silently between runs of the same notebook.
GENEFORMER_PIN = "04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && git checkout -q {GENEFORMER_PIN}
!cd {GENEFORMER_REPO} && pip install .
!pip uninstall -y torchao -q      # unused; hard-raises inside peft dispatch below its floor

GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True).stdout.strip()
assert GENEFORMER_COMMIT == GENEFORMER_PIN, (
    f"Geneformer clone is at {GENEFORMER_COMMIT}, expected pinned {GENEFORMER_PIN}")

import torch
assert torch.cuda.is_available(), "no CUDA GPU -- select a GPU runtime before embedding"
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT,
      "| GPU:", torch.cuda.get_device_name(0))

SMOKE   = False
SMOKE_N_PER_GROUP = 25          # cells kept per (cell_type x donor) group when SMOKE
SUFFIX  = "_SMOKE" if SMOKE else ""
SEED    = 0
from datetime import date
TODAY   = date.today().isoformat()
print(f"SMOKE={SMOKE} | output suffix='{SUFFIX}'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing /content/Geneformer
  Preparing metadata (setup.py) ... done
  Created wheel for geneformer: filename=geneformer-0.1.0-py3-none-any.whl size=2980779 sha256=abfe2bdc3d64a90b8d6a38c327c1da9b935f1e56a2b53c9a99ba101ca9bbd211
  Stored in directory: /tmp/pip-ephem-wheel-cache-adfbx_d6/wheels/2d/46/09/b7648deddd8f78de5e5a2785bbf00a6b4d1246c6d434192a76
Successfully built geneformer
  Attempting uninstall: geneformer
    Found existing installation: geneformer 0.1.0
    Uninstalling geneformer-0.1.0:
      Successfully uninstalled geneformer-0.1.0
Python: 3.12.13 | Geneformer commit: 04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5 | GPU: NVIDIA A100-SXM4-80GB
SMOKE=False | output suffix=''


> **Interpretation — environment confirmed, real run started (1a).** The pinned Geneformer clone resolved to and was asserted against `04c2b2e8...`, the exact commit used throughout this project — an unpinned clone would let both the library code and the V2-104M weights drift silently between runs, which pip's own pins can't catch since they never see this HuggingFace clone. The GPU resolved to an A100-SXM4-80GB, and `SMOKE=False | output suffix=''` confirms this is the real reference run, not a plumbing dry run: every path this notebook writes from here on is the permanent, non-suffixed location, and the adaptive batch-size fix added to section 4a after this notebook's earlier crashes is about to be exercised on the full reference for the first time.

## 2 — Acquire and prepare the non-AD reference

### 2a — Load the reference and validate its schema

The forgetting probe needs an out-of-domain dataset with (i) **raw counts** (Geneformer rank-encodes raw counts) and (ii) trustworthy **cell-type labels** to classify. The reference source is isolated in the config below so it can be swapped without touching anything downstream.

**Default = PBMC** (a labeled, raw-counts peripheral-blood dataset): small, canonical, reproducible, contract-sanctioned. **Known limitation, reported not hidden:** microglia are myeloid, so PBMC's monocyte/DC compartment overlaps the lineage CPT trained on — PBMC can *under-detect* forgetting there; its lymphoid (T/B/NK) types are genuine out-of-domain and carry the probe. **Stricter alternative = Tabula Sapiens** (multi-organ breadth, maximally distant from glia): swap `REF_*` below for a Tabula Sapiens h5ad and the rest of the notebook is unchanged.

`REF_H5AD` is the one value to confirm before running — point it at a real labeled raw-counts h5ad. The asserts below fail loud if raw counts or cell-type labels are missing rather than silently embedding garbage.

In [2]:
import numpy as np, pandas as pd, anndata as ad, scanpy as sc, scipy.sparse as sp
sc.settings.verbosity = 1

# --- reference config (the one block to edit to swap references) ---
REF_H5AD        = os.path.join(DRIVE_ROOT, "reference", "pbmc_reference.h5ad")  # Kang et al. 2018 (GSE96583)
REF_CELLTYPE_COL = "cell_type"     # obs column holding the cell-type labels to classify
REF_DONOR_COL    = "replicate"     # obs column for donor-disjoint split; set None if absent (Kang 2018: 8 patient donors)
REF_RAW_LAYER    = None            # layer holding raw counts; None => use .X (or .raw if .X is normalized)
REF_NAME         = "pbmc"          # tag used in output filenames / audit
REF_SOURCE_URL   = "https://exampledata.scverse.org/pertpy/kang_2018.h5ad"  # only used to auto-fetch the pbmc default

if not os.path.exists(REF_H5AD) and REF_NAME == "pbmc":
    print("reference not found on Drive -- downloading Kang et al. 2018 (GSE96583, non-AD PBMC reference) ...")
    os.makedirs(os.path.dirname(REF_H5AD), exist_ok=True)
    import urllib.request, shutil
    tmp_path = REF_H5AD + ".part"
    # The host sits behind Cloudflare, which 403s urllib's default `Python-urllib/x.y` User-Agent
    # (verified 2026-07-21: Python-urllib -> 403; browser/curl/requests UAs -> 200). Send a UA
    # explicitly. `urlopen` also replaces `urlretrieve`, which is a deprecated legacy interface.
    req = urllib.request.Request(REF_SOURCE_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(tmp_path, "wb") as fh:
        expected = resp.headers.get("Content-Length")
        shutil.copyfileobj(resp, fh)
    got = os.path.getsize(tmp_path)
    if expected is not None and got != int(expected):
        os.remove(tmp_path)   # never leave a truncated file that a later run would treat as complete
        raise RuntimeError(f"truncated download: got {got} bytes, expected {expected}")
    os.rename(tmp_path, REF_H5AD)   # atomic: a crash mid-download leaves .part, never a truncated "real" file
    print(f"downloaded -> {REF_H5AD} ({got/1e6:.1f} MB)")

assert os.path.exists(REF_H5AD), (
    f"reference not found: {REF_H5AD}. Place a labeled raw-counts non-AD h5ad there "
    "(PBMC default, or a Tabula Sapiens h5ad for the stricter breadth probe).")

ref = sc.read_h5ad(REF_H5AD)
print("reference:", ref.shape)

# resolve raw counts
if REF_RAW_LAYER is not None:
    assert REF_RAW_LAYER in ref.layers, f"layer '{REF_RAW_LAYER}' absent; layers={list(ref.layers)}"
    ref.X = ref.layers[REF_RAW_LAYER].copy()
def _int_frac(X):
    idx = np.random.default_rng(0).choice(X.shape[0], size=min(2000, X.shape[0]), replace=False)
    d = X[idx]; d = d.data if sp.issparse(d) else np.asarray(d).ravel()
    return float(np.mean(np.mod(d, 1) == 0)) if d.size else 0.0
if _int_frac(ref.X) < 0.99 and ref.raw is not None:
    print("`.X` looks normalized -- falling back to `.raw` for raw counts")
    ref = ref.raw.to_adata()
assert _int_frac(ref.X) >= 0.99, "reference `.X` is not raw counts (needed for Geneformer)"

# validate labels
assert REF_CELLTYPE_COL in ref.obs.columns, f"reference lacks cell-type column '{REF_CELLTYPE_COL}'"
ref.obs["cell_type"] = ref.obs[REF_CELLTYPE_COL].astype(str)
assert ref.obs["cell_type"].nunique() >= 5, (
    f"only {ref.obs['cell_type'].nunique()} cell types -- too few for a meaningful forgetting probe")
HAS_DONOR = REF_DONOR_COL is not None and REF_DONOR_COL in ref.obs.columns
ref.obs["donor_id"] = ref.obs[REF_DONOR_COL].astype(str) if HAS_DONOR else "single"
print(f"cell types: {ref.obs['cell_type'].nunique()} | donors: {ref.obs['donor_id'].nunique()} "
      f"(donor-disjoint split: {HAS_DONOR})")
print(ref.obs["cell_type"].value_counts().to_string())

reference: (24673, 15706)
cell types: 8 | donors: 8 (donor-disjoint split: True)
cell_type
CD4 T cells          11238
CD14+ Monocytes       5697
B cells               2651
NK cells              1716
CD8 T cells           1621
FCGR3A+ Monocytes     1089
Dendritic cells        529
Megakaryocytes         132


> **Interpretation — reference loaded and validated (2a).** The Kang et al. 2018 PBMC reference (GSE96583) loaded at 24,673 cells × 15,706 genes. All three schema asserts passed silently: `.X` is ≥99% integer-valued (raw counts, not normalized), the `cell_type` column is present with 8 distinct labels (well above the 5-type floor), and the `replicate` column (Kang's real patient donor field, wired in as `REF_DONOR_COL`) is present with 8 distinct donors — so `HAS_DONOR=True` and the split in section 5 will be donor-disjoint rather than a weaker per-cell stratified split. The printed composition is heavily imbalanced: CD4 T cells alone is 11,238 cells (~46% of the reference), while Megakaryocytes is just 132 (~0.5%) — worth flagging now because that thin tail is exactly what trips the test-set floor in 5a later. The composition also previews this reference's known asymmetry as a forgetting probe: CD14+/FCGR3A+ Monocytes and Dendritic cells are myeloid, the same broad lineage as the microglia CPT trained on, so any forgetting there could be under-detected; the T/B/NK lymphoid types are genuinely out-of-domain and carry the probe's real sensitivity.

### 2b — Cap per cell type, map genes to the Geneformer vocabulary

The reference is capped per cell type (balanced, up to a ceiling) so no single abundant type dominates the k-NN and so the four embedding passes stay tractable. A stable `ref_index` is assigned as the realignment key — after the `SMOKE` subsample, so it numbers the rows actually embedded. Genes are mapped to Ensembl and vocabulary coverage is reported — unlike colab_12 there is **no APOE hard-fail gate** (that is glia/eval-#2 specific); the forgetting probe only needs broad coverage, so low niche-gene coverage is irrelevant here.

In [3]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
import pickle

CAP_PER_TYPE = 3000
rng = np.random.default_rng(SEED)
keep_idx = []
for ct, sub in ref.obs.groupby("cell_type", observed=True):
    pos = np.where(ref.obs["cell_type"].values == ct)[0]
    keep_idx.append(pos if len(pos) <= CAP_PER_TYPE else rng.choice(pos, CAP_PER_TYPE, replace=False))
keep_idx = np.sort(np.concatenate(keep_idx))
ref = ref[keep_idx].copy()
print(f"after per-type cap ({CAP_PER_TYPE}): {ref.n_obs} cells across {ref.obs['cell_type'].nunique()} types")

# --- SMOKE subsample: BEFORE tokenization/embedding, so a plumbing run is actually cheap ---
# Thinned per (cell_type x donor) rather than per cell_type: the 5a split is donor-disjoint,
# so every donor and every type must survive or the split logic is not exercised at all.
# Placed ahead of the ref_index assignment below, which is the realignment key the four
# embedding passes reindex on -- it has to number the rows that are actually embedded.
if SMOKE:
    _rng = np.random.default_rng(SEED)
    _ctv, _dnv = ref.obs["cell_type"].values, ref.obs["donor_id"].values
    _keep = []
    for _c in pd.unique(_ctv):
        for _d in pd.unique(_dnv):
            _p = np.where((_ctv == _c) & (_dnv == _d))[0]
            if len(_p) == 0:
                continue
            _keep.append(_p if len(_p) <= SMOKE_N_PER_GROUP
                         else _rng.choice(_p, SMOKE_N_PER_GROUP, replace=False))
    ref = ref[np.sort(np.concatenate(_keep))].copy()
    print(f"[SMOKE] subsampled to {ref.n_obs} cells | "
          f"types {ref.obs['cell_type'].nunique()} | donors {ref.obs['donor_id'].nunique()}")

ref.obs["ref_index"] = np.arange(ref.n_obs)

with open(ENSEMBL_DICTIONARY_FILE, "rb") as f: symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICTIONARY_FILE, "rb") as f: token_dictionary = pickle.load(f)
# reference var_names assumed to be gene SYMBOLS; if they are already Ensembl IDs, set ensembl_id directly.
ref.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in ref.var_names]
in_vocab = ref.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)
frac = float(in_vocab.mean())
print(f"genes in Geneformer vocab: {int(in_vocab.sum())} / {ref.n_vars} ({frac:.1%})")
assert frac >= 0.30, (
    f"only {frac:.1%} of reference genes are tokenizable -- var_names may be Ensembl IDs already "
    "(set ref.var['ensembl_id'] = ref.var_names) or the wrong ID type")

after per-type cap (3000): 13738 cells across 8 types
genes in Geneformer vocab: 12231 / 15706 (77.9%)


> **Interpretation — capped and vocabulary-mapped (2b).** The per-type cap (3,000) only bit on the two largest classes: CD4 T cells (11,238 → 3,000) and CD14+ Monocytes (5,697 → 3,000); the other 6 types were already below the ceiling and passed through untouched. That brings the working set to 13,738 cells across the same 8 types — every type is still represented, just less lopsidedly, which matters for the balanced-accuracy k-NN read in section 5. The `SMOKE` subsample block was skipped entirely since `SMOKE=False`. Gene-to-Ensembl mapping then found 12,231 of 15,706 reference genes (77.9%) fall inside Geneformer's tokenizer vocabulary — comfortably clearing the 30% sanity floor. Unlike the glia notebooks, there is deliberately no hard-fail gate on any specific gene here (no APOE-style assert): this reference only needs broad enough coverage for a generic cell-type read, not any one niche gene to survive tokenization.

## 3 — Tokenize the reference

### 3a — Tokenize with the same encoding as every other Geneformer notebook

Same V2 / 4096-input encoding and the same `sum_ensembl_ids` RangeIndex monkeypatch colab_11/12 needed. The cell-type, donor, and `ref_index` labels are carried into every tokenized cell so the embeddings realign afterwards.

In [4]:
from geneformer import TranscriptomeTokenizer
import geneformer.tokenizer as _gftok

ref.obs["n_counts"] = np.asarray(ref.X.sum(axis=1)).ravel()
assert (ref.obs["n_counts"] > 0).all(), "reference cells with zero counts present"

TOK_IN_DIR  = f"/content/gf_ref_token_in{SUFFIX}"
TOK_OUT_DIR = f"/content/gf_ref_token_out{SUFFIX}"
os.makedirs(TOK_IN_DIR, exist_ok=True); os.makedirs(TOK_OUT_DIR, exist_ok=True)
ref.write_h5ad(os.path.join(TOK_IN_DIR, "ref.h5ad"))

CUSTOM_ATTRS = {c: c for c in ["ref_index", "cell_type", "donor_id"]}

_orig_sum = _gftok.sum_ensembl_ids
def _sum_rangeindex_patch(*a, **k):
    r = _orig_sum(*a, **k); r.var.index = pd.RangeIndex(r.n_vars); return r
_gftok.sum_ensembl_ids = _sum_rangeindex_patch

tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, f"ref_{REF_NAME}{SUFFIX}", file_format="h5ad")
TOKENIZED_REF = os.path.join(TOK_OUT_DIR, f"ref_{REF_NAME}{SUFFIX}.dataset")
print("tokenized reference ->", TOKENIZED_REF)

Tokenizing /content/gf_ref_token_in/ref.h5ad
/content/gf_ref_token_in/ref.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
tokenized reference -> /content/gf_ref_token_out/ref_pbmc.dataset


> **Interpretation — reference tokenized (3a).** Tokenization used the same `sum_ensembl_ids` RangeIndex monkeypatch colab_11/12 needed, over the same V2/4096-token encoding as every Geneformer notebook in this project. The message "no column attribute 'filter_pass' -- tokenizing all cells" is expected and benign: it just means no cells were pre-flagged for exclusion, so all 13,738 post-cap cells went into the tokenized dataset, none silently dropped. `ref_index`, `cell_type`, and `donor_id` were carried through as custom attributes on every tokenized cell — this is the realignment key that lets the four embedding passes in 4a reindex their output rows back onto the exact same cells in the same order.

## 4 — Embed the reference: zero-shot and CPT, at both extraction points

### 4a — Four embedding passes (base × v2) × (`emb_layer=-1` × `emb_layer=0`)

The frozen base gives the zero-shot embeddings; the base with the v2 LoRA adapter merged in gives the post-CPT embeddings. Each is extracted at both the second-to-last layer (−1, the pipeline readout) and the last layer (0). Four aligned matrices over the same reference cells, each cached to Drive and exists-guarded so a re-run skips the GPU work.

In [5]:
from peft import PeftModel
from transformers import BertForMaskedLM
from geneformer import EmbExtractor

# transformers 4.57 defaults BERT to SDPA attention. On this Colab image (torch 2.11.0+cu128,
# driver 580.82.07 / CUDA 13.0) BOTH fused SDPA backends fault on the V2-104M forward -- flash
# and mem-efficient alike -- with a device-side assert / illegal memory access, while eager runs
# clean. Verified 2026-07-21 by A/B on identical batches: eager OK, sdpa fails; generic matmul
# and nn.Embedding(20275,768) on GPU both pass; the same forward is correct on CPU. So this is a
# fused-kernel fault, not data, weights, or config -- the tokenized input was independently
# checked (max token id 20239 < vocab 20275, max len 2516 < 4096, no empty rows, length column
# exact). geneformer.perturber_utils.load_model builds model_args without attn_implementation
# and resolves BertForMaskedLM from module globals at call time, so patch it there.
import geneformer.perturber_utils as _pu

class _EagerBertForMaskedLM:
    @staticmethod
    def from_pretrained(*a, **k):
        k.setdefault("attn_implementation", "eager")
        return BertForMaskedLM.from_pretrained(*a, **k)

_pu.BertForMaskedLM = _EagerBertForMaskedLM

GF_DIR      = os.path.join(DRIVE_ROOT, "geneformer")
REF_DIR     = os.path.join(DRIVE_ROOT, "reference")
os.makedirs(REF_DIR, exist_ok=True)
MODEL_DIR   = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
ADAPTER_DIR = os.path.join(GF_DIR, "cpt_aggregated_v2_seed0_adapter")   # colab_11 v2 output
assert os.path.exists(MODEL_DIR),   f"base model missing: {MODEL_DIR}"
assert os.path.exists(ADAPTER_DIR), f"v2 LoRA adapter missing on Drive: {ADAPTER_DIR}"

# --- batch size is bounded by the attention tensor, not by GPU memory ---
# get_embs sorts longest-sequence-first, so batch 0 is the widest the run will ever be, and eager
# attention materialises a (B, heads, L, L) score tensor for it. Once that exceeds ~2^31 elements
# the CUDA kernels indexing it overflow signed-32-bit and fault as a device-side assert / illegal
# memory access -- which surfaces at the first kernel of the forward (the embedding lookup), not
# at the attention op itself. That is the crash signature seen here on 2026-07-21. It is purely a
# geometry bound: at B=64, heads=12 the widest safe sequence is 1672 tokens, and this reference
# tokenizes to 2516. Halve B until the batch fits. Embeddings are batch-invariant in eval mode
# (padding is masked, cells never interact), so this changes cost and nothing else.
from datasets import load_from_disk
from transformers import BertConfig

_cfg     = BertConfig.from_pretrained(MODEL_DIR)
MAX_LEN  = int(max(load_from_disk(TOKENIZED_REF)["length"]))
N_HEADS  = _cfg.num_attention_heads
INT32MAX = 2**31 - 1

assert MAX_LEN <= _cfg.max_position_embeddings, (
    f"tokenized length {MAX_LEN} exceeds max_position_embeddings {_cfg.max_position_embeddings}")

FWD_BATCH = 64
while FWD_BATCH > 1 and FWD_BATCH * N_HEADS * MAX_LEN**2 > INT32MAX:
    FWD_BATCH //= 2
print(f"max tokenized length {MAX_LEN} (limit {_cfg.max_position_embeddings}) | heads {N_HEADS}")
print(f"forward_batch_size {FWD_BATCH} -> attention tensor {FWD_BATCH*N_HEADS*MAX_LEN**2:,} "
      f"elements vs int32 limit {INT32MAX:,}")
if FWD_BATCH < 64:
    print(f"  NOTE: reduced from 64; {64*N_HEADS*MAX_LEN**2:,} elements would have overflowed")

EMB_LABELS = ["ref_index", "cell_type", "donor_id"]
LAYER_TAG  = {-1: "L-1", 0: "L0"}

def _ref_emb_path(model_tag, layer):
    # cap/seed baked into the filename so a re-run with changed params can never silently
    # load a stale cached embedding keyed only on model_tag/layer
    return os.path.join(REF_DIR,
        f"ref_{REF_NAME}_{model_tag}_{LAYER_TAG[layer].replace('-','m')}_cap{CAP_PER_TYPE}_seed{SEED}{SUFFIX}.h5ad")

def _extract(model_dir, model_tag, layer):
    out = _ref_emb_path(model_tag, layer)
    if os.path.exists(out):
        print(f"{model_tag} {LAYER_TAG[layer]} exists, loading:", out)
        a = sc.read_h5ad(out)
    else:
        ee = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                          max_ncells=None, emb_layer=layer, emb_label=EMB_LABELS,
                          forward_batch_size=FWD_BATCH, nproc=4)
        work = f"/content/gf_ref_emb_{model_tag}_{LAYER_TAG[layer]}{SUFFIX}"; os.makedirs(work, exist_ok=True)
        df = ee.extract_embs(model_dir, TOKENIZED_REF, work, f"ref_{model_tag}_{LAYER_TAG[layer]}")
        emb_cols = [c for c in df.columns if c not in EMB_LABELS]
        df = df.set_index("ref_index").reindex(ref.obs["ref_index"].values)
        assert df[emb_cols].notna().all().all(), f"{model_tag} {LAYER_TAG[layer]}: rows missing after realignment"
        a = ad.AnnData(X=df[emb_cols].to_numpy(dtype=np.float32),
                       obs=ref.obs[EMB_LABELS].reset_index(drop=True))
        a.write_h5ad(out)
        print(f"{model_tag} {LAYER_TAG[layer]} -> {out}  {a.shape}")
    return a.X.astype(np.float32)

# zero-shot: frozen base at both layers
EMB = {}
for layer in (-1, 0):
    EMB[("zeroshot", LAYER_TAG[layer])] = _extract(MODEL_DIR, "zeroshot", layer)

# CPT: merge v2 adapter once, then both layers
MERGED_DIR = f"/content/gf_v2_merged{SUFFIX}"
if not os.path.exists(os.path.join(MERGED_DIR, "config.json")):
    base = BertForMaskedLM.from_pretrained(MODEL_DIR)
    merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    os.makedirs(MERGED_DIR, exist_ok=True)
    merged.save_pretrained(MERGED_DIR)
    base.config.to_json_file(os.path.join(MERGED_DIR, "config.json"))
for layer in (-1, 0):
    EMB[("cpt", LAYER_TAG[layer])] = _extract(MERGED_DIR, "cpt", layer)

for k, v in EMB.items(): print(k, v.shape)
assert len({v.shape for v in EMB.values()}) == 1, "reference embedding matrices differ in shape"

max tokenized length 2516 (limit 4096) | heads 12
forward_batch_size 16 -> attention tensor 1,215,409,152 elements vs int32 limit 2,147,483,647
  NOTE: reduced from 64; 4,861,636,608 elements would have overflowed


/usr/local/lib/python3.12/dist-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


  0%|          | 0/859 [00:00<?, ?it/s]

/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


zeroshot L-1 -> /content/drive/MyDrive/ad-glia-fm-prep/reference/ref_pbmc_zeroshot_Lm1_cap3000_seed0.h5ad  (13738, 768)


...

/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


cpt L-1 -> /content/drive/MyDrive/ad-glia-fm-prep/reference/ref_pbmc_cpt_Lm1_cap3000_seed0.h5ad  (13738, 768)


  0%|          | 0/859 [00:00<?, ?it/s]

cpt L0 -> /content/drive/MyDrive/ad-glia-fm-prep/reference/ref_pbmc_cpt_L0_cap3000_seed0.h5ad  (13738, 768)
('zeroshot', 'L-1') (13738, 768)
('zeroshot', 'L0') (13738, 768)
('cpt', 'L-1') (13738, 768)
('cpt', 'L0') (13738, 768)


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


> **Interpretation — the fixed crash path ran clean on the full reference (4a).** This is the section that faulted twice before on this notebook (device-side assert / illegal memory access, traced to the eager-attention score tensor overflowing signed int32 at wide batches). The printed numbers show the fix engaging exactly as designed: max tokenized length came out to 2,516 (well under the 4,096 position-embedding limit, so that isn't the constraint), and with 12 attention heads a batch of 64 would need 4,861,636,608 attention-tensor elements — over double the int32 ceiling of 2,147,483,647, which is what actually overflowed before. The adaptive logic halved the batch to 16, bringing the tensor to 1,215,409,152 elements, safely under the limit. This is the first time that fix has been exercised on the real, full 13,738-cell reference rather than the earlier small dry run. All four passes then completed cleanly — zero-shot at both layers, then the v2 LoRA adapter merged once and reused for both CPT-layer passes — each landing at exactly `(13738, 768)`, confirming every one of the 13,738 cells got a real embedding vector in all four matrices (a mismatch here would have failed the realignment assert inside `_extract`). The repeated CLS/EOS warnings are Geneformer's own expected behavior for `emb_mode="cell"` (mean-pooling excludes those two special tokens from the average, not a sign of anything wrong). Wall-clock for all four passes together was about 11 minutes — well under even a conservative estimate scaled from this project's one prior real-embedding timing anchor (colab_12's ~1h43m/pass at 142,588 cells), plausibly because this reference's shorter, more uniform per-cell sequences make each pass cheaper per cell, though that specific cause wasn't independently verified. Running both models at both layers (rather than just the pipeline's usual `emb_layer=-1`) is deliberate: the head-ablation finding placed CPT's loss-fit gain downstream of the −1 readout, so a forgetting probe blind to L0 could miss damage concentrated exactly where CPT's changes are known to live.

## 5 — Eval #3 / Detector #2: k-NN cell-type forgetting

### 5a — Reference split and composition (donor-disjoint where possible; rare types excluded)

A held-out split of the reference: donor-disjoint if the reference carries donors, else stratified by cell type. Because zero-shot and CPT are scored on the *same* split, any residual leakage affects both equally and cancels in the drop. Cell types too rare in aggregate are dropped before the split. Separately, cell types that end up too thin specifically *in the held-out test split* — a real risk with a handful of donors and uneven per-donor composition, since a donor-disjoint split can't stratify by cell type — are excluded from k-NN scoring after the split, so an unreliable per-class accuracy estimate can never inject noise into the drop that feeds detector #2's gate. A type the probe cannot reliably see is not evidence of forgetting either way.

In [6]:
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit

ct = ref.obs["cell_type"].to_numpy()
dn = ref.obs["donor_id"].to_numpy()

# --- drop rare cell types before splitting ---
MIN_CELLS_PER_TYPE = 50
vc = pd.Series(ct).value_counts()
keep_types = set(vc[vc >= MIN_CELLS_PER_TYPE].index)
dropped = sorted(set(vc.index) - keep_types)
keep = np.isin(ct, list(keep_types))
print(f"cell types kept: {len(keep_types)} | dropped (<{MIN_CELLS_PER_TYPE} cells): {dropped}")

idx = np.where(keep)[0]
if ref.obs["donor_id"].nunique() > 3:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr_rel, te_rel = next(gss.split(idx, ct[idx], groups=dn[idx]))
    split_kind = "donor-disjoint"
else:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr_rel, te_rel = next(sss.split(idx, ct[idx]))
    split_kind = "stratified-by-celltype"
train_idx, test_idx = idx[tr_rel], idx[te_rel]

# (SMOKE thinning happens at 2b, before tokenization/embedding -- see the note there.)

print(f"split: {split_kind} | train {len(train_idx)} / test {len(test_idx)} cells")
print("test cell-type counts:")
print(pd.Series(ct[test_idx]).value_counts().to_string())

# --- exclude cell types too thin in the TEST split to score reliably ---
# The MIN_CELLS_PER_TYPE floor above only guards aggregate counts, drawn before the split.
# A donor-disjoint split over this few donors (and per-donor composition this uneven) can
# still leave a type with a handful of test cells even though it cleared that floor.
# balanced_accuracy_score weights every class equally regardless of size, so a thin class
# injects noise into the zero-shot-vs-CPT drop that does NOT cancel between the two --
# exactly the kind of instability a hard pass/fail gate (detector #2) shouldn't be exposed to.
MIN_TEST_PER_TYPE = 20
test_vc = pd.Series(ct[test_idx]).value_counts()
underpowered_types = sorted(test_vc[test_vc < MIN_TEST_PER_TYPE].index)
if underpowered_types:
    print(f"\nWARNING: {len(underpowered_types)} type(s) have <{MIN_TEST_PER_TYPE} test cells "
          f"after the split -- excluding from k-NN scoring: {underpowered_types}")
    train_idx = train_idx[~np.isin(ct[train_idx], underpowered_types)]
    test_idx  = test_idx[~np.isin(ct[test_idx],  underpowered_types)]
scored_types = sorted(keep_types - set(underpowered_types))
print(f"\nfinal: train {len(train_idx)} / test {len(test_idx)} cells, {len(scored_types)} types scored: {scored_types}")

cell types kept: 8 | dropped (<50 cells): []
split: donor-disjoint | train 6111 / test 7627 cells
test cell-type counts:
CD14+ Monocytes      1663
B cells              1510
CD8 T cells          1384
CD4 T cells          1287
NK cells              970
FCGR3A+ Monocytes     597
Dendritic cells       205
Megakaryocytes         11


final: train 5990 / test 7616 cells, 7 types scored: ['B cells', 'CD14+ Monocytes', 'CD4 T cells', 'CD8 T cells', 'Dendritic cells', 'FCGR3A+ Monocytes', 'NK cells']


> **Interpretation — split drawn, one type excluded from scoring (5a).** With 8 donors (above the >3 threshold), the split defaulted to donor-disjoint (`GroupShuffleSplit`, `test_size=0.30` — a fraction of *donors*, not cells, so the resulting cell-level split isn't literally 70/30), giving a raw train/test of 6,111/7,627 cells (≈44/56 by cell count, since donors vary widely in size). The aggregate `MIN_CELLS_PER_TYPE=50` floor dropped nothing — all 8 types cleared it. But the printed test-set composition shows why a second, post-split guard is needed: Megakaryocytes had 132 cells overall (well above 50) yet only 11 landed in the test fold — a donor-disjoint split with just 8 donors and this uneven a per-donor composition can strand a technically-adequate type in a tiny test slice purely by which donors happened to fall on which side. The `MIN_TEST_PER_TYPE=20` guard caught this and excluded Megakaryocytes from both train and test for every layer's scoring, leaving 5,990/7,616 cells across 7 types. This matters specifically because `balanced_accuracy_score` weights every class equally regardless of size — a noisy 11-cell estimate wouldn't cancel between the zero-shot and CPT comparison and would inject unreliable noise straight into detector #2's hard pass/fail gate. The 7 types that remain are comfortably powered for the k-NN read that follows (205–1,663 test cells each).

### 5b — k-NN cell-type accuracy, zero-shot vs CPT, at both layers → forgetting

For each extraction point, a k-NN is fit on the train cells' embedding to predict cell type and scored (balanced accuracy across types) on the held-out cells — once on the zero-shot embedding, once on the CPT embedding. The **drop** = zero-shot − CPT (positive = forgetting) is scored against eval #3's bands and detector #2's gate. Reading both layers shows whether any forgetting sits at the pipeline readout (−1) or concentrates in the last layer (0) where v2's changes are known to live.

In [7]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

def band_forget(drop_pp):        # eval #3 reporting bands (drop = zs - cpt, positive = worse)
    if drop_pp > 15: return "catastrophic"
    if drop_pp > 5:  return "concerning"
    return "acceptable"
def gate_detector2(drop_pp):     # detector #2 hard gate
    return "OVERTRAIN" if drop_pp > 20 else "pass"

def knn_bacc(X):
    sc_ = StandardScaler().fit(X[train_idx])
    knn = KNeighborsClassifier(n_neighbors=15).fit(sc_.transform(X[train_idx]), ct[train_idx])
    return balanced_accuracy_score(ct[test_idx], knn.predict(sc_.transform(X[test_idx])))

if underpowered_types:
    print(f"NOTE: {len(underpowered_types)} type(s) excluded from every layer's scoring for thin "
          f"test representation (<{MIN_TEST_PER_TYPE} cells): {underpowered_types}\n")

EVAL3 = {}
for layer in ("L-1", "L0"):
    acc_zs  = knn_bacc(EMB[("zeroshot", layer)])
    acc_cpt = knn_bacc(EMB[("cpt", layer)])
    drop = (acc_zs - acc_cpt) * 100
    EVAL3[layer] = {"acc_zeroshot": round(float(acc_zs), 4), "acc_cpt": round(float(acc_cpt), 4),
                    "drop_pp": round(float(drop), 2), "eval3_verdict": band_forget(drop),
                    "detector2_gate": gate_detector2(drop),
                    "n_types_scored": len(scored_types), "excluded_types": underpowered_types}
    print(f"{layer}: zero-shot {acc_zs:.3f} -> CPT {acc_cpt:.3f} | drop {drop:+.2f} pp | "
          f"eval#3 {band_forget(drop)} | detector#2 {gate_detector2(drop)}")

# primary gate = pipeline readout (-1); flag if L0 is materially worse than -1
GATE_PRIMARY = EVAL3["L-1"]["detector2_gate"]
L0_WORSE = EVAL3["L0"]["drop_pp"] - EVAL3["L-1"]["drop_pp"] > 5
print(f"\ndetector #2 (primary, L-1): {GATE_PRIMARY}"
      + ("  | NOTE: L0 forgetting exceeds L-1 by >5pp -- deeper-layer damage the readout hides" if L0_WORSE else ""))

NOTE: 1 type(s) excluded from every layer's scoring for thin test representation (<20 cells): ['Megakaryocytes']

L-1: zero-shot 0.883 -> CPT 0.882 | drop +0.09 pp | eval#3 acceptable | detector#2 pass
L0: zero-shot 0.933 -> CPT 0.931 | drop +0.24 pp | eval#3 acceptable | detector#2 pass

detector #2 (primary, L-1): pass


> **Interpretation — no detectable forgetting, at either layer (5b).** For each extraction point, a k-NN (k=15, features standardized on train only) was fit on the 5,990 train cells' embedding to predict cell type, then scored by balanced accuracy on the 7,616 held-out cells — once on the zero-shot embedding, once on the CPT embedding, with `drop_pp` = zero-shot − CPT (positive = CPT is worse). Results: L−1 88.3% → 88.2% (drop +0.09pp), L0 93.3% → 93.0% (drop +0.24pp) — both far inside eval #3's "acceptable" band (≤5pp) and nowhere near detector #2's hard >20pp overtrain gate. The L0-vs-L−1 check (does the deeper layer show forgetting the standard readout would miss?) also didn't trigger — L0's drop isn't more than 5pp worse than L−1's. Worth being precise about what this does and doesn't show: unlike colab_12's APOE k-NN, which sat at or below chance regardless of CPT (a floor null that can't distinguish "no effect" from "nothing there to detect"), both baselines here are well above chance (~14% for 7 balanced classes) at 88–93% — this probe had real discriminative structure to lose, and CPT didn't cost it any. Combined with colab_12: CPT's loss-fit gain, already established as concentrated in the head and last encoder layer, isn't recoverable as substate/APOE structure through either accessible extraction point — but it also isn't damaging the model's general cell-type knowledge on this reference. Still N=1 pilot, and PBMC's myeloid overlap with microglia (CD14+/FCGR3A+ Monocytes, Dendritic cells) means this reference could under-detect forgetting specifically in that compartment even though the aggregate k-NN read is clean; Tabula Sapiens remains the stricter breadth check if that overlap is ever a concern.

## 6 — Summary, gate on colab_12, and handoff

### 6a — Apply detector #2 to colab_12's wins, append the audit trace, print commit commands

Detector #2's verdict is applied back to colab_12: if it tripped, any eval #1/#2 win recorded under `geneformer_cpt_evals` is re-labelled **overtrain-confounded**. If colab_12 has not been run yet, that is noted and the gate is deferred. The forgetting results are written under `geneformer_cpt_forgetting`. A `SMOKE` run writes nothing to the audit trace.

In [9]:
import json, shlex

print("=== EVAL #3 / DETECTOR #2 (forgetting on non-AD reference: %s) ===" % REF_NAME)
for layer, r in EVAL3.items():
    print(f"  {layer:4s}: {r['acc_zeroshot']:.3f} -> {r['acc_cpt']:.3f} | drop {r['drop_pp']:+.2f}pp "
          f"| eval#3 {r['eval3_verdict']} | detector#2 {r['detector2_gate']}")

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

# gate colab_12's recorded wins
overtrain = GATE_PRIMARY == "OVERTRAIN"
gate_note = []
if "geneformer_cpt_evals" in report:
    e = report["geneformer_cpt_evals"]
    wins = []
    for blk in ("eval1_substate_probe", "eval2_apoe_recovery"):
        for key, r in e.get(blk, {}).items():
            for vk in ("verdict", "knn_verdict"):
                if r.get(vk) in ("meaningful", "decisive"):
                    wins.append(f"{blk}:{key}:{vk}={r[vk]}")
    if overtrain and wins:
        gate_note = [f"OVERTRAIN-CONFOUNDED (detector#2 tripped): {w}" for w in wins]
        print("\n[GATE] detector #2 tripped -- colab_12 wins re-labelled overtrain-confounded:")
        for w in wins: print("   -", w)
    elif wins:
        gate_note = [f"win stands (detector#2 pass): {w}" for w in wins]
        print(f"\n[GATE] detector #2 pass -- {len(wins)} colab_12 win(s) stand.")
    else:
        print("\n[GATE] colab_12 recorded no eval wins to gate.")
else:
    gate_note = ["colab_12 (geneformer_cpt_evals) not yet in audit -- apply gate once it runs"]
    print("\n[GATE] colab_12 evals not yet recorded -- gate deferred until it runs.")

if SMOKE:
    print("\n[SMOKE] audit trace NOT written (plumbing run).")
else:
    report["geneformer_cpt_forgetting"] = {
        "status": "computed", "date": TODAY, "reads_run": "geneformer_cpt_aggregated_v2",
        "reference": REF_NAME, "reference_file": os.path.relpath(REF_H5AD, DRIVE_ROOT),
        "n_ref_cells": int(ref.n_obs), "n_cell_types": int(ref.obs["cell_type"].nunique()),
        "geneformer_commit": GENEFORMER_COMMIT,
        "extraction_points": {"L-1": "second-to-last", "L0": "last layer"},
        "eval3_detector2": EVAL3, "detector2_primary_gate": GATE_PRIMARY,
        "colab_12_gate": gate_note,
        "note": ("eval#3 and detector#2 are one k-NN read two ways. PBMC's myeloid overlap with microglia "
                 "may under-detect forgetting; Tabula Sapiens is the stricter breadth swap."),
    }
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("\naudit trace appended ->", AUDIT_REPORT_PATH)
    rel = os.path.relpath(AUDIT_REPORT_PATH, REPO_PATH)
    print("\n=== Commit + push (from WSL) ===")
    print(f"  git add {shlex.quote(rel)}")
    print("  git commit -m 'colab_13: Geneformer CPT eval #3 / detector #2 (forgetting) at L-1 and L0'")
    print("  git push")

=== EVAL #3 / DETECTOR #2 (forgetting on non-AD reference: pbmc) ===
  L-1 : 0.883 -> 0.882 | drop +0.09pp | eval#3 acceptable | detector#2 pass
  L0  : 0.933 -> 0.930 | drop +0.24pp | eval#3 acceptable | detector#2 pass

[GATE] colab_12 recorded no eval wins to gate.

audit trace appended -> /content/ad-glia-fm-prep/outputs/audit_report.json

=== Commit + push (from WSL) ===
  git add outputs/audit_report.json
  git commit -m 'colab_13: Geneformer CPT eval #3 / detector #2 (forgetting) at L-1 and L0'
  git push


> **Interpretation — gate applied, audit trace written (6a).** This cell re-reads `outputs/audit_report.json` and checks colab_12's `geneformer_cpt_evals` block for any substate or APOE verdict recorded as `meaningful` or `decisive` — the kind of result detector #2 tripping would force a re-label on. Colab_12 recorded none (both its evals were null at every extraction point and lineage), so the print correctly reports "colab_12 recorded no eval wins to gate": there is nothing for this run's `pass` verdict to act on. Because `SMOKE=False`, the full `geneformer_cpt_forgetting` block — reference name and file, cell/type counts, the pinned Geneformer commit, both layers' zero-shot/CPT accuracies and drops, the primary gate, and the (empty) gate note — was appended to the JSON and written back to disk. That JSON entry, not this notebook's live session, is the durable record that makes this run's result checkable later. The printed `git add`/`commit`/`push` lines are informational only: Colab has no push credentials for this repo, so the actual commit of the updated `audit_report.json` happens separately from a machine that does.